# TAHAP 03 — Text Preprocessing & Feature Engineering

## Tujuan Tahap 03

Tahap ini bertujuan untuk menyiapkan data teks agar dapat digunakan pada tahap pemodelan berikutnya, yaitu intent classification dan recommender system. Fokus utama tahap ini adalah membersihkan teks, menormalisasi variasi bahasa, melakukan tokenisasi, menghapus stopword, melakukan stemming jika library tersedia, serta membangun fitur gabungan berbasis teks untuk program studi, karier, skill, dan roadmap belajar.

## Output Utama

1. `data/processed/intent_dataset_processed.csv`
2. `data/processed/program_studi_s1_processed.csv`
3. `data/processed/karier_processed.csv`
4. `data/processed/skill_awal_processed.csv`
5. `data/processed/roadmap_belajar_processed.csv`
6. `data/processed/normalisasi_bahasa_processed.csv`
7. `reports/data_quality/preprocessing_quality_summary.csv`
8. `reports/data_quality/preprocessing_samples.csv`
9. `reports/data_quality/feature_engineering_summary.csv`

## 1. Import Library dan Setup Path Project

Cell ini mengatur library utama, mencari root project, dan membuat folder output apabila belum tersedia.


In [1]:
from pathlib import Path
from datetime import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)


def find_project_root(start: Path | None = None) -> Path:
    """
    Mencari root project berdasarkan keberadaan folder data/interim.
    Aman digunakan ketika notebook dijalankan dari folder notebooks.
    """
    start = (start or Path.cwd()).resolve()
    candidates = [start] + list(start.parents)

    for candidate in candidates:
        if (candidate / "data" / "interim").exists():
            return candidate

    # Fallback: jika cwd adalah notebooks, root project adalah parent-nya.
    if start.name.lower() == "notebooks":
        return start.parent

    return start


PROJECT_ROOT = find_project_root()
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DATA_QUALITY_DIR = PROJECT_ROOT / "reports" / "data_quality"

DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DATA_QUALITY_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT              :", PROJECT_ROOT)
print("DATA_INTERIM_DIR          :", DATA_INTERIM_DIR)
print("DATA_PROCESSED_DIR        :", DATA_PROCESSED_DIR)
print("REPORTS_DATA_QUALITY_DIR  :", REPORTS_DATA_QUALITY_DIR)

if not DATA_INTERIM_DIR.exists():
    raise FileNotFoundError(
        f"Folder data/interim tidak ditemukan: {DATA_INTERIM_DIR}\n"
        "Pastikan notebook dijalankan dari folder notebooks atau root project, "
        "dan output Tahap 02 sudah tersedia di data/interim."
    )

PROJECT_ROOT              : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant
DATA_INTERIM_DIR          : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\interim
DATA_PROCESSED_DIR        : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\processed
REPORTS_DATA_QUALITY_DIR  : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\data_quality


## 2. Load Dataset dari `data/interim`

Dataset yang digunakan merupakan hasil validasi Tahap 02.

In [2]:
FILES = {
    "intent": "intent_dataset_interim.csv",
    "program": "program_studi_s1_interim.csv",
    "career": "karier_interim.csv",
    "skill": "skill_awal_interim.csv",
    "roadmap": "roadmap_belajar_interim.csv",
    "normalization": "normalisasi_bahasa_interim.csv",
}


def load_csv(file_name: str) -> pd.DataFrame:
    path = DATA_INTERIM_DIR / file_name
    if not path.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {path}")
    return pd.read_csv(path)


intent_df = load_csv(FILES["intent"])
program_df = load_csv(FILES["program"])
career_df = load_csv(FILES["career"])
skill_df = load_csv(FILES["skill"])
roadmap_df = load_csv(FILES["roadmap"])
normalization_df = load_csv(FILES["normalization"])

datasets = {
    "intent": intent_df,
    "program": program_df,
    "career": career_df,
    "skill": skill_df,
    "roadmap": roadmap_df,
    "normalization": normalization_df,
}

overview_df = pd.DataFrame([
    {"dataset": name, "rows": df.shape[0], "columns": df.shape[1]}
    for name, df in datasets.items()
])

display(overview_df)

,dataset,rows,columns
0,intent,12,16
1,program,5,19
2,career,5,12
3,skill,7,12
4,roadmap,6,12
5,normalization,18,8


## 3. Audit Awal Data Teks

Audit dilakukan untuk melihat missing value, duplikasi teks, dan panjang teks awal. Pada dataset intent, kolom utama yang diproses adalah `utterance`.

In [3]:
def audit_text_column(df: pd.DataFrame, dataset_name: str, text_col: str) -> dict:
    if text_col not in df.columns:
        return {
            "dataset": dataset_name,
            "text_column": text_col,
            "status": "missing_column",
            "rows": len(df),
            "missing_text": np.nan,
            "empty_text": np.nan,
            "duplicate_text": np.nan,
            "avg_text_length": np.nan,
            "min_text_length": np.nan,
            "max_text_length": np.nan,
        }

    text_series = df[text_col].fillna("").astype(str).str.strip()
    lengths = text_series.str.len()

    return {
        "dataset": dataset_name,
        "text_column": text_col,
        "status": "ok",
        "rows": len(df),
        "missing_text": int(df[text_col].isna().sum()),
        "empty_text": int((text_series == "").sum()),
        "duplicate_text": int(text_series.duplicated().sum()),
        "avg_text_length": round(float(lengths.mean()), 2) if len(lengths) else 0,
        "min_text_length": int(lengths.min()) if len(lengths) else 0,
        "max_text_length": int(lengths.max()) if len(lengths) else 0,
    }


text_audit_df = pd.DataFrame([
    audit_text_column(intent_df, "intent", "utterance"),
    audit_text_column(program_df, "program", "deskripsi_singkat"),
    audit_text_column(career_df, "career", "deskripsi_singkat"),
    audit_text_column(skill_df, "skill", "deskripsi_singkat"),
    audit_text_column(roadmap_df, "roadmap", "tujuan_fase"),
])

display(text_audit_df)

print("Contoh utterance awal:")
display(intent_df[["utterance_id", "utterance", "intent_label", "bahasa", "register"]].head(10))

,dataset,text_column,status,rows,missing_text,empty_text,duplicate_text,avg_text_length,min_text_length,max_text_length
0,intent,utterance,ok,12,0,0,0,54.92,32,84
1,program,deskripsi_singkat,ok,5,0,0,0,98.00,89,113
2,career,deskripsi_singkat,ok,5,0,0,0,64.60,52,78
3,skill,deskripsi_singkat,ok,7,0,0,0,83.14,68,93
4,roadmap,tujuan_fase,ok,6,0,0,0,50.33,38,59


Contoh utterance awal:


,utterance_id,utterance,intent_label,bahasa,register
0,UTT001,"Saya suka matematika dan ingin bekerja di bidang data, program studi apa yang cocok?",rekomendasi_prodi,id_formal,formal
1,UTT002,"Aku seneng ngoding sama statistik, cocoknya kuliah apa ya?",rekomendasi_prodi,id_nonformal,nonformal
2,UTT003,"Aku pengin dadi analis data, jurusan sing pas apa?",rekomendasi_prodi,jawa_umum,nonformal
3,UTT004,"Abdi resep komputer jeung angka, prodi naon nu cocog?",rekomendasi_prodi,sunda_umum,nonformal
4,UTT005,"Kalau ingin masuk Teknik Informatika, saya harus belajar apa dulu?",roadmap_belajar,id_formal,formal
5,UTT006,Lulusan Sistem Informasi biasanya bisa bekerja sebagai apa?,prospek_karier,id_formal,formal
6,UTT007,Skill awal untuk jadi data analyst itu apa saja?,skill_awal,id_nonformal,nonformal
7,UTT008,Apa perbedaan Teknik Informatika dan Sistem Informasi?,info_program_studi,id_formal,formal
8,UTT009,"Halo, saya mau konsultasi jurusan kuliah.",sapaan,id_formal,formal
9,UTT010,Saya bingung banget dan belum tahu minat saya apa.,klarifikasi_minat,id_nonformal,nonformal


## 4. Fungsi Dasar Text Preprocessing

Urutan preprocessing pada tahap ini:

1. **Case folding**: mengubah teks menjadi huruf kecil.
2. **Cleaning karakter khusus**: menghapus URL, simbol, tanda baca berlebih, dan spasi ganda.
3. **Normalisasi bahasa**: mengganti kata non-formal, Jawa umum, dan Sunda umum menjadi bentuk baku.
4. **Tokenization**: memecah teks menjadi token kata.
5. **Stopword removal**: menghapus kata umum yang kurang informatif.
6. **Stemming**: mengubah kata menjadi bentuk dasar jika library tersedia.

In [4]:
def safe_text(value) -> str:
    """Mengubah nilai menjadi string yang aman diproses."""
    if pd.isna(value):
        return ""
    return str(value)


def case_fold(text: str) -> str:
    """Mengubah teks menjadi huruf kecil dan menghapus spasi awal/akhir."""
    return safe_text(text).lower().strip()


def clean_special_chars(text: str) -> str:
    """
    Membersihkan URL, karakter khusus, tanda baca, dan spasi berlebih.
    Angka tetap dipertahankan karena dapat relevan untuk konteks kelas atau durasi belajar.
    """
    text = case_fold(text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text: str) -> list[str]:
    """Tokenisasi sederhana berbasis regex."""
    return re.findall(r"\b[a-z0-9]+\b", safe_text(text).lower())


# Uji cepat fungsi dasar
sample_text = "Aku seneng ngoding!!! Cocoknya kuliah apa ya?"
print("Original :", sample_text)
print("Clean    :", clean_special_chars(sample_text))
print("Tokens   :", tokenize(clean_special_chars(sample_text)))

Original : Aku seneng ngoding!!! Cocoknya kuliah apa ya?
Clean    : aku seneng ngoding cocoknya kuliah apa ya
Tokens   : ['aku', 'seneng', 'ngoding', 'cocoknya', 'kuliah', 'apa', 'ya']


## 5. Membangun Kamus Normalisasi Bahasa

Kamus normalisasi berasal dari `normalisasi_bahasa_interim.csv`. Kamus ini penting karena input user dapat berupa Bahasa Indonesia formal, non-formal, Jawa umum, dan Sunda umum.

In [5]:
required_norm_cols = {"kata_input", "kata_baku", "prioritas"}
missing_norm_cols = required_norm_cols - set(normalization_df.columns)
if missing_norm_cols:
    raise ValueError(f"Kolom normalisasi tidak lengkap: {missing_norm_cols}")

# Bersihkan kolom kamus agar konsisten dengan proses cleaning utama.
norm_work = normalization_df.copy()
norm_work["kata_input_clean"] = norm_work["kata_input"].apply(clean_special_chars)
norm_work["kata_baku_clean"] = norm_work["kata_baku"].apply(clean_special_chars)

# Prioritas lebih kecil dianggap lebih penting.
norm_work = norm_work.sort_values(by=["prioritas", "kata_input_clean"], ascending=[True, True])

normalization_map = {}
for _, row in norm_work.iterrows():
    key = row["kata_input_clean"]
    value = row["kata_baku_clean"]
    if key and value and key not in normalization_map:
        normalization_map[key] = value

print("Jumlah entri kamus normalisasi:", len(normalization_map))
print("Contoh mapping:")
for i, (key, value) in enumerate(normalization_map.items()):
    if i >= 12:
        break
    print(f"- {key} -> {value}")


def normalize_tokens(tokens: list[str], norm_map: dict[str, str]) -> list[str]:
    """Normalisasi token berdasarkan kamus. Jika value berupa frasa, akan dipecah lagi pada tahap tokenize berikutnya."""
    return [norm_map.get(token, token) for token in tokens]


def normalize_text(text: str, norm_map: dict[str, str]) -> str:
    """Normalisasi teks berbasis token exact match."""
    tokens = tokenize(clean_special_chars(text))
    normalized_tokens = normalize_tokens(tokens, norm_map)
    normalized_text = " ".join(normalized_tokens)
    normalized_text = re.sub(r"\s+", " ", normalized_text).strip()
    return normalized_text


sample_text = "Aku seneng ngoding, pengin dadi analis data. Prodi naon nu cocog?"
print("\nOriginal    :", sample_text)
print("Clean       :", clean_special_chars(sample_text))
print("Normalized  :", normalize_text(sample_text, normalization_map))
print("Tokens norm :", tokenize(normalize_text(sample_text, normalization_map)))

Jumlah entri kamus normalisasi: 18
Contoh mapping:
- dadi -> menjadi
- gue -> saya
- gw -> saya
- hoyong -> ingin
- janten -> menjadi
- kumaha -> bagaimana
- naon -> apa
- ngoding -> pemrograman
- opo -> apa
- pengin -> ingin
- pgn -> ingin
- pingin -> ingin

Original    : Aku seneng ngoding, pengin dadi analis data. Prodi naon nu cocog?
Clean       : aku seneng ngoding pengin dadi analis data prodi naon nu cocog
Normalized  : saya suka pemrograman ingin menjadi analis data prodi apa nu cocog
Tokens norm : ['saya', 'suka', 'pemrograman', 'ingin', 'menjadi', 'analis', 'data', 'prodi', 'apa', 'nu', 'cocog']


## 6. Stopword Removal Bahasa Indonesia

Stopword adalah kata umum yang sering muncul tetapi tidak selalu membawa informasi kuat untuk klasifikasi atau rekomendasi. Pada project ini, beberapa kata domain seperti `data`, `program`, `studi`, `informatika`, `karier`, dan `skill` tetap dipertahankan karena relevan terhadap konteks EduPath.

In [6]:
DEFAULT_INDONESIAN_STOPWORDS = {
    "ada", "adalah", "agar", "akan", "aku", "anda", "apa", "apakah", "atau", "bagi", "bagaimana",
    "bagai", "banyak", "begitu", "belum", "bisa", "buat", "cara", "dalam", "dan", "dari", "dengan",
    "di", "dia", "dulu", "hal", "harus", "ini", "itu", "jadi", "jika", "juga", "kalau", "kami",
    "kamu", "kan", "karena", "ke", "kepada", "ketika", "lagi", "lebih", "maka", "mau", "mereka",
    "mohon", "namun", "nya", "oleh", "pada", "paling", "para", "per", "saja", "saling", "sama",
    "saya", "sebagai", "sebelum", "sebuah", "secara", "sedang", "sehingga", "seperti", "serta",
    "si", "supaya", "tapi", "telah", "tentang", "tersebut", "tidak", "untuk", "ya", "yang"
}

try:
    from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
    stopword_factory = StopWordRemoverFactory()
    stopwords = set(stopword_factory.get_stop_words()) | DEFAULT_INDONESIAN_STOPWORDS
    STOPWORD_SOURCE = "Sastrawi + custom"
except Exception as e:
    stopwords = set(DEFAULT_INDONESIAN_STOPWORDS)
    STOPWORD_SOURCE = f"custom fallback; Sastrawi StopWord tidak tersedia ({type(e).__name__})"

CUSTOM_EXTRA_STOPWORDS = {
    "jeung", "nu", "mah", "teh", "banget", "dong", "nih", "sih", "ya", "aja", "cocoknya", "pas"
}

DOMAIN_KEEP_WORDS = {
    "data", "program", "studi", "prodi", "kuliah", "jurusan", "karier", "skill", "roadmap",
    "informatika", "sistem", "informasi", "sains", "bisnis", "komputer", "statistik", "analisis",
    "analis", "developer", "programmer", "software", "engineer", "machine", "learning", "dashboard",
    "matematika", "desain", "komunikasi", "akuntansi", "manajemen"
}

stopwords = (stopwords | CUSTOM_EXTRA_STOPWORDS) - DOMAIN_KEEP_WORDS

print("Sumber stopword:", STOPWORD_SOURCE)
print("Jumlah stopword aktif:", len(stopwords))
print("Contoh stopword:", sorted(list(stopwords))[:30])

Sumber stopword: Sastrawi + custom
Jumlah stopword aktif: 158
Contoh stopword: ['ada', 'adalah', 'agak', 'agar', 'aja', 'akan', 'aku', 'amat', 'anda', 'antara', 'anu', 'apa', 'apakah', 'apalagi', 'atau', 'bagai', 'bagaimana', 'bagaimanapun', 'bagi', 'bahwa', 'banget', 'banyak', 'begitu', 'belum', 'bisa', 'boleh', 'buat', 'cara', 'cocoknya', 'dahulu']


## 7. Stemming Bahasa Indonesia Jika Library Tersedia

Stemming digunakan untuk mengubah kata berimbuhan menjadi bentuk dasar. Contoh: `menganalisis` dapat menjadi `analisis`. Jika `Sastrawi` belum tersedia, notebook tetap berjalan tanpa stemming.

In [7]:
try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    stemmer_factory = StemmerFactory()
    stemmer = stemmer_factory.create_stemmer()
    STEMMING_AVAILABLE = True
    STEMMER_SOURCE = "Sastrawi"
except Exception as e:
    stemmer = None
    STEMMING_AVAILABLE = False
    STEMMER_SOURCE = f"tidak tersedia ({type(e).__name__}); proses stemming dilewati"


def stem_tokens(tokens: list[str]) -> list[str]:
    """Melakukan stemming jika stemmer tersedia."""
    if not STEMMING_AVAILABLE:
        return tokens
    return [stemmer.stem(token) for token in tokens]


print("Status stemming:", STEMMER_SOURCE)
print("Contoh stemming:", stem_tokens(["menganalisis", "pemrograman", "pembelajaran", "rekomendasi"]))

Status stemming: Sastrawi
Contoh stemming: ['analis', 'pemrograman', 'ajar', 'rekomendasi']


## 8. Pipeline Preprocessing Utama

Cell ini membangun kolom:

1. `utterance_clean`
2. `utterance_normalized`
3. `utterance_tokens`
4. `utterance_preprocessed`

In [8]:
def preprocess_text(text: str, norm_map: dict[str, str], apply_stemming: bool = True) -> dict:
    """Pipeline lengkap preprocessing teks."""
    clean_text = clean_special_chars(text)
    normalized_text = normalize_text(clean_text, norm_map)
    tokens = tokenize(normalized_text)

    tokens_no_stopword = [
        token for token in tokens
        if token not in stopwords and len(token) > 1
    ]

    if apply_stemming:
        final_tokens = stem_tokens(tokens_no_stopword)
    else:
        final_tokens = tokens_no_stopword

    final_tokens = [token for token in final_tokens if token and len(token) > 1]
    preprocessed_text = " ".join(final_tokens)

    return {
        "clean": clean_text,
        "normalized": normalized_text,
        "tokens": tokens,
        "tokens_no_stopword": tokens_no_stopword,
        "tokens_final": final_tokens,
        "preprocessed": preprocessed_text,
    }


processed_utterances = intent_df["utterance"].apply(
    lambda text: preprocess_text(text, normalization_map, apply_stemming=True)
)

intent_df["utterance_clean"] = processed_utterances.apply(lambda x: x["clean"])
intent_df["utterance_normalized"] = processed_utterances.apply(lambda x: x["normalized"])
intent_df["utterance_tokens"] = processed_utterances.apply(lambda x: json.dumps(x["tokens"], ensure_ascii=False))
intent_df["utterance_preprocessed"] = processed_utterances.apply(lambda x: x["preprocessed"])
intent_df["utterance_token_count"] = processed_utterances.apply(lambda x: len(x["tokens_final"]))

sample_cols = [
    "utterance_id", "utterance", "utterance_clean", "utterance_normalized",
    "utterance_tokens", "utterance_preprocessed", "intent_label", "bahasa"
]

display(intent_df[sample_cols].head(12))

,utterance_id,utterance,utterance_clean,utterance_normalized,utterance_tokens,utterance_preprocessed,intent_label,bahasa
0,UTT001,"Saya suka matematika dan ingin bekerja di bidang data, program studi apa yang cocok?",saya suka matematika dan ingin bekerja di bidang data program studi apa yang cocok,saya suka matematika dan ingin bekerja di bidang data program studi apa yang cocok,"[""saya"", ""suka"", ""matematika"", ""dan"", ""ingin"", ""bekerja"", ""di"", ""bidang"", ""data"", ""program"", ""studi"", ""apa"", ""yang"",...",suka matematika kerja bidang data program studi cocok,rekomendasi_prodi,id_formal
1,UTT002,"Aku seneng ngoding sama statistik, cocoknya kuliah apa ya?",aku seneng ngoding sama statistik cocoknya kuliah apa ya,saya suka pemrograman sama statistik cocoknya kuliah apa ya,"[""saya"", ""suka"", ""pemrograman"", ""sama"", ""statistik"", ""cocoknya"", ""kuliah"", ""apa"", ""ya""]",suka pemrograman statistik kuliah,rekomendasi_prodi,id_nonformal
2,UTT003,"Aku pengin dadi analis data, jurusan sing pas apa?",aku pengin dadi analis data jurusan sing pas apa,saya ingin menjadi analis data program studi yang pas apa,"[""saya"", ""ingin"", ""menjadi"", ""analis"", ""data"", ""program"", ""studi"", ""yang"", ""pas"", ""apa""]",jadi analis data program studi,rekomendasi_prodi,jawa_umum
3,UTT004,"Abdi resep komputer jeung angka, prodi naon nu cocog?",abdi resep komputer jeung angka prodi naon nu cocog,abdi suka komputer jeung angka prodi apa nu cocog,"[""abdi"", ""suka"", ""komputer"", ""jeung"", ""angka"", ""prodi"", ""apa"", ""nu"", ""cocog""]",abdi suka komputer angka prodi cocog,rekomendasi_prodi,sunda_umum
4,UTT005,"Kalau ingin masuk Teknik Informatika, saya harus belajar apa dulu?",kalau ingin masuk teknik informatika saya harus belajar apa dulu,kalau ingin masuk teknik informatika saya harus belajar apa dulu,"[""kalau"", ""ingin"", ""masuk"", ""teknik"", ""informatika"", ""saya"", ""harus"", ""belajar"", ""apa"", ""dulu""]",masuk teknik informatika ajar,roadmap_belajar,id_formal
5,UTT006,Lulusan Sistem Informasi biasanya bisa bekerja sebagai apa?,lulusan sistem informasi biasanya bisa bekerja sebagai apa,lulusan sistem informasi biasanya bisa bekerja sebagai apa,"[""lulusan"", ""sistem"", ""informasi"", ""biasanya"", ""bisa"", ""bekerja"", ""sebagai"", ""apa""]",lulus sistem informasi biasa kerja,prospek_karier,id_formal
6,UTT007,Skill awal untuk jadi data analyst itu apa saja?,skill awal untuk jadi data analyst itu apa saja,skill awal untuk jadi data analyst itu apa saja,"[""skill"", ""awal"", ""untuk"", ""jadi"", ""data"", ""analyst"", ""itu"", ""apa"", ""saja""]",skill awal data analyst,skill_awal,id_nonformal
7,UTT008,Apa perbedaan Teknik Informatika dan Sistem Informasi?,apa perbedaan teknik informatika dan sistem informasi,apa perbedaan teknik informatika dan sistem informasi,"[""apa"", ""perbedaan"", ""teknik"", ""informatika"", ""dan"", ""sistem"", ""informasi""]",beda teknik informatika sistem informasi,info_program_studi,id_formal
8,UTT009,"Halo, saya mau konsultasi jurusan kuliah.",halo saya mau konsultasi jurusan kuliah,halo saya mau konsultasi program studi kuliah,"[""halo"", ""saya"", ""mau"", ""konsultasi"", ""program"", ""studi"", ""kuliah""]",halo konsultasi program studi kuliah,sapaan,id_formal
9,UTT010,Saya bingung banget dan belum tahu minat saya apa.,saya bingung banget dan belum tahu minat saya apa,saya bingung banget dan belum tahu minat saya apa,"[""saya"", ""bingung"", ""banget"", ""dan"", ""belum"", ""tahu"", ""minat"", ""saya"", ""apa""]",bingung tahu minat,klarifikasi_minat,id_nonformal


## 9. Feature Engineering untuk Recommender System

Pada tahap ini dibuat teks gabungan untuk profil program studi, karier, skill, dan roadmap. Fitur gabungan ini akan berguna pada Tahap 04 dan 05 ketika membangun recommender system berbasis TF-IDF dan cosine similarity.

In [13]:
def join_columns(row: pd.Series, columns: list[str]) -> str:
    """Menggabungkan beberapa kolom teks menjadi satu dokumen profil."""
    values = []
    for col in columns:
        if col in row.index:
            value = row[col]
            if pd.notna(value):
                value = str(value).replace("|", " ").strip()
                value = re.sub(r"\s+", " ", value)
                if value and value.lower() not in {"nan", "none", "null"}:
                    values.append(value)
    return " ".join(values)


def create_profile_text(
    df: pd.DataFrame,
    source_columns: list[str],
    output_column: str,
    raw_output_column: str | None = None
) -> pd.DataFrame:
    """Membuat raw profile text dan profile text yang sudah dipreproses."""
    df = df.copy()
    raw_col = raw_output_column or f"{output_column}_raw"
    df[raw_col] = df.apply(lambda row: join_columns(row, source_columns), axis=1)
    df[output_column] = df[raw_col].apply(
        lambda text: preprocess_text(text, normalization_map, apply_stemming=True)["preprocessed"]
    )
    df[f"{output_column}_token_count"] = df[output_column].apply(lambda text: len(tokenize(text)))
    return df


program_profile_cols = [
    "nama_program_studi", "jenjang", "rumpun_ilmu", "bidang_keilmuan", "deskripsi_singkat",
    "minat_cocok", "mapel_relevan", "hobi_relevan", "gaya_belajar_cocok", "karakteristik_cocok",
    "tingkat_matematika", "tingkat_coding", "tingkat_komunikasi", "tingkat_desain", "keyword_rekomendasi"
]

career_profile_cols = [
    "nama_karier", "bidang_karier", "deskripsi_singkat", "tugas_utama", "skill_teknis",
    "skill_non_teknis", "tools_umum", "prospek_industri", "keyword_karier"
]

skill_profile_cols = [
    "nama_skill", "kategori_skill", "deskripsi_singkat", "level_awal", "prasyarat",
    "sumber_belajar_awal", "estimasi_waktu_belajar", "keyword_skill"
]

roadmap_profile_cols = [
    "fase", "durasi_rekomendasi", "tujuan_fase", "materi_pokok", "aktivitas_praktik",
    "output_portofolio", "tools_umum", "indikator_capaian"
]

program_df = create_profile_text(program_df, program_profile_cols, "program_profile_text")
career_df = create_profile_text(career_df, career_profile_cols, "career_profile_text")
skill_df = create_profile_text(skill_df, skill_profile_cols, "skill_profile_text")
roadmap_df = create_profile_text(roadmap_df, roadmap_profile_cols, "roadmap_profile_text")

print("Contoh program_profile_text:")
display(program_df[["program_id", "nama_program_studi", "program_profile_text"]].head())

print("Contoh career_profile_text:")
display(career_df[["karier_id", "nama_karier", "career_profile_text"]].head())

print("Contoh skill_profile_text:")
display(skill_df[["skill_id", "nama_skill", "skill_profile_text"]].head())

Contoh program_profile_text:


,program_id,nama_program_studi,program_profile_text
0,PRG001,Teknik Informatika,teknik informatika s1 komputer software engineering komputasi ajar pemrograman algoritma basis data jaring kembang p...
1,PRG002,Sistem Informasi,sistem informasi s1 komputer sistem bisnis teknologi informasi ajar integrasi teknologi proses bisnis analisis butuh...
2,PRG003,Sains Data,sains data s1 komputer data science analitik ajar statistika pemrograman machine learning visualisasi data ambil put...
3,PRG004,Statistika,statistika s1 matematika sains statistika terap ajar kumpul data probabilitas model statistik survei analisis kuanti...
4,PRG005,Desain Komunikasi Visual,desain komunikasi visual s1 seni desain desain visual komunikasi digital ajar desain grafis branding ilustrasi ui ko...


Contoh career_profile_text:


,karier_id,nama_karier,career_profile_text
0,KAR001,Data Analyst,data analyst data analitik analis data hasil insight bisnis bersih data buat dashboard susun lapor insight excel sql...
1,KAR002,Data Scientist,data scientist data ai bangun model analitik machine learning prediksi rekomendasi eda feature engineering modeling ...
2,KAR003,Software Engineer,software engineer software development kembang aplikasi web mobile api sistem perangkat lunak coding testing debuggi...
3,KAR004,UI/UX Designer,ui ux designer desain produk digital rancang tampil alam guna aplikasi user research wireframe prototype testing fig...
4,KAR005,Business Analyst,business analyst bisnis teknologi analis butuh bisnis terjemah jadi solusi sistem requirement gathering process mapp...


Contoh skill_profile_text:


,skill_id,nama_skill,skill_profile_text
0,SKL001,Dasar Pemrograman Python,dasar pemrograman python technical paham variabel tipe data cabang ulang fungsi struktur data sederhana mula logika ...
1,SKL002,Dasar Basis Data dan SQL,dasar basis data sql technical paham tabel relasi query select filter join agregasi kelola data sederhana mula logik...
2,SKL003,Analisis Data Dasar,analisis data dasar technical paham baca data statistik deskriptif pola data insight sederhana mula matematika dasar...
3,SKL004,Algoritma dan Struktur Data Dasar,algoritma struktur data dasar technical paham problem solving array list searching sorting dasar mula pemrograman da...
4,SKL005,Komunikasi dan Analisis Kebutuhan,komunikasi analisis butuh soft skill mampu gali butuh buat tanya susun catat jelas solusi mula mampu komunikasi dasa...


## 10. Validasi Hasil Preprocessing

Validasi dilakukan untuk memastikan kolom baru tidak kosong, token terbentuk, dan fitur profil siap digunakan untuk tahap pemodelan.

In [14]:
def validate_columns_not_empty(df: pd.DataFrame, dataset_name: str, columns: list[str]) -> list[dict]:
    records = []
    for col in columns:
        if col not in df.columns:
            records.append({
                "dataset": dataset_name,
                "column": col,
                "status": "missing_column",
                "rows": len(df),
                "missing_count": np.nan,
                "empty_count": np.nan,
                "valid_count": np.nan,
            })
            continue

        series = df[col].fillna("").astype(str).str.strip()
        missing_count = int(df[col].isna().sum())
        empty_count = int((series == "").sum())
        valid_count = int((series != "").sum())
        status = "ok" if empty_count == 0 else "check_empty"

        records.append({
            "dataset": dataset_name,
            "column": col,
            "status": status,
            "rows": len(df),
            "missing_count": missing_count,
            "empty_count": empty_count,
            "valid_count": valid_count,
        })
    return records


validation_records = []
validation_records += validate_columns_not_empty(
    intent_df,
    "intent",
    ["utterance_clean", "utterance_normalized", "utterance_tokens", "utterance_preprocessed"]
)
validation_records += validate_columns_not_empty(program_df, "program", ["program_profile_text"])
validation_records += validate_columns_not_empty(career_df, "career", ["career_profile_text"])
validation_records += validate_columns_not_empty(skill_df, "skill", ["skill_profile_text"])
validation_records += validate_columns_not_empty(roadmap_df, "roadmap", ["roadmap_profile_text"])

preprocessing_validation_df = pd.DataFrame(validation_records)
display(preprocessing_validation_df)

intent_length_summary = intent_df[["utterance", "utterance_preprocessed", "utterance_token_count"]].copy()
intent_length_summary["original_length"] = intent_length_summary["utterance"].fillna("").astype(str).str.len()
intent_length_summary["preprocessed_length"] = intent_length_summary["utterance_preprocessed"].fillna("").astype(str).str.len()

display(intent_length_summary.describe(include="all"))

,dataset,column,status,rows,missing_count,empty_count,valid_count
0,intent,utterance_clean,ok,12,0,0,12
1,intent,utterance_normalized,ok,12,0,0,12
2,intent,utterance_tokens,ok,12,0,0,12
3,intent,utterance_preprocessed,ok,12,0,0,12
4,program,program_profile_text,ok,5,0,0,5
5,career,career_profile_text,ok,5,0,0,5
6,skill,skill_profile_text,ok,7,0,0,7
7,roadmap,roadmap_profile_text,ok,6,0,0,6


,utterance,utterance_preprocessed,utterance_token_count,original_length,preprocessed_length
count,12,12,12.000000,12.000000,12.000000
unique,12,12,NaN,NaN,NaN
top,"Saya suka matematika dan ingin bekerja di bidang data, program studi apa yang cocok?",suka matematika kerja bidang data program studi cocok,NaN,NaN,NaN
freq,1,1,NaN,NaN,NaN
mean,NaN,NaN,5.166667,54.916667,33.833333
std,NaN,NaN,1.403459,13.131907,8.840335
min,NaN,NaN,3.000000,32.000000,18.000000
25%,NaN,NaN,4.000000,49.500000,29.750000
50%,NaN,NaN,5.000000,53.500000,34.000000
75%,NaN,NaN,6.000000,60.250000,37.000000


## 11. Membuat Laporan Ringkas Preprocessing

Laporan ini disimpan ke folder `reports/data_quality` sebagai dokumentasi Tahap 03.

In [15]:
run_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

preprocessing_quality_summary = pd.DataFrame([
    {
        "run_timestamp": run_timestamp,
        "dataset": "intent",
        "rows": len(intent_df),
        "new_columns": "utterance_clean|utterance_normalized|utterance_tokens|utterance_preprocessed",
        "empty_utterance_preprocessed": int((intent_df["utterance_preprocessed"].fillna("").astype(str).str.strip() == "").sum()),
        "avg_final_token_count": round(float(intent_df["utterance_token_count"].mean()), 2),
        "stemming_available": STEMMING_AVAILABLE,
        "stopword_source": STOPWORD_SOURCE,
        "normalization_terms": len(normalization_map),
    },
    {
        "run_timestamp": run_timestamp,
        "dataset": "program",
        "rows": len(program_df),
        "new_columns": "program_profile_text",
        "empty_profile_text": int((program_df["program_profile_text"].fillna("").astype(str).str.strip() == "").sum()),
        "avg_profile_token_count": round(float(program_df["program_profile_text_token_count"].mean()), 2),
        "stemming_available": STEMMING_AVAILABLE,
        "stopword_source": STOPWORD_SOURCE,
        "normalization_terms": len(normalization_map),
    },
    {
        "run_timestamp": run_timestamp,
        "dataset": "career",
        "rows": len(career_df),
        "new_columns": "career_profile_text",
        "empty_profile_text": int((career_df["career_profile_text"].fillna("").astype(str).str.strip() == "").sum()),
        "avg_profile_token_count": round(float(career_df["career_profile_text_token_count"].mean()), 2),
        "stemming_available": STEMMING_AVAILABLE,
        "stopword_source": STOPWORD_SOURCE,
        "normalization_terms": len(normalization_map),
    },
    {
        "run_timestamp": run_timestamp,
        "dataset": "skill",
        "rows": len(skill_df),
        "new_columns": "skill_profile_text",
        "empty_profile_text": int((skill_df["skill_profile_text"].fillna("").astype(str).str.strip() == "").sum()),
        "avg_profile_token_count": round(float(skill_df["skill_profile_text_token_count"].mean()), 2),
        "stemming_available": STEMMING_AVAILABLE,
        "stopword_source": STOPWORD_SOURCE,
        "normalization_terms": len(normalization_map),
    },
    {
        "run_timestamp": run_timestamp,
        "dataset": "roadmap",
        "rows": len(roadmap_df),
        "new_columns": "roadmap_profile_text",
        "empty_profile_text": int((roadmap_df["roadmap_profile_text"].fillna("").astype(str).str.strip() == "").sum()),
        "avg_profile_token_count": round(float(roadmap_df["roadmap_profile_text_token_count"].mean()), 2),
        "stemming_available": STEMMING_AVAILABLE,
        "stopword_source": STOPWORD_SOURCE,
        "normalization_terms": len(normalization_map),
    },
])

preprocessing_samples = intent_df[[
    "utterance_id", "utterance", "utterance_clean", "utterance_normalized",
    "utterance_tokens", "utterance_preprocessed", "intent_label", "bahasa", "register"
]].copy()

feature_engineering_summary = pd.DataFrame([
    {
        "dataset": "program",
        "feature_column": "program_profile_text",
        "source_columns": "|".join([col for col in program_profile_cols if col in program_df.columns]),
        "rows": len(program_df),
        "empty_feature": int((program_df["program_profile_text"].fillna("").astype(str).str.strip() == "").sum()),
    },
    {
        "dataset": "career",
        "feature_column": "career_profile_text",
        "source_columns": "|".join([col for col in career_profile_cols if col in career_df.columns]),
        "rows": len(career_df),
        "empty_feature": int((career_df["career_profile_text"].fillna("").astype(str).str.strip() == "").sum()),
    },
    {
        "dataset": "skill",
        "feature_column": "skill_profile_text",
        "source_columns": "|".join([col for col in skill_profile_cols if col in skill_df.columns]),
        "rows": len(skill_df),
        "empty_feature": int((skill_df["skill_profile_text"].fillna("").astype(str).str.strip() == "").sum()),
    },
    {
        "dataset": "roadmap",
        "feature_column": "roadmap_profile_text",
        "source_columns": "|".join([col for col in roadmap_profile_cols if col in roadmap_df.columns]),
        "rows": len(roadmap_df),
        "empty_feature": int((roadmap_df["roadmap_profile_text"].fillna("").astype(str).str.strip() == "").sum()),
    },
])

display(preprocessing_quality_summary)
display(feature_engineering_summary)

,run_timestamp,dataset,rows,new_columns,empty_utterance_preprocessed,avg_final_token_count,stemming_available,stopword_source,normalization_terms,empty_profile_text,avg_profile_token_count
0,2026-06-14 14:01:42,intent,12,utterance_clean|utterance_normalized|utterance_tokens|utterance_preprocessed,0.0,5.17,True,Sastrawi + custom,18,NaN,NaN
1,2026-06-14 14:01:42,program,5,program_profile_text,NaN,NaN,True,Sastrawi + custom,18,0.0,44.80
2,2026-06-14 14:01:42,career,5,career_profile_text,NaN,NaN,True,Sastrawi + custom,18,0.0,36.40
3,2026-06-14 14:01:42,skill,7,skill_profile_text,NaN,NaN,True,Sastrawi + custom,18,0.0,26.86
4,2026-06-14 14:01:42,roadmap,6,roadmap_profile_text,NaN,NaN,True,Sastrawi + custom,18,0.0,31.33


,dataset,feature_column,source_columns,rows,empty_feature
0,program,program_profile_text,nama_program_studi|jenjang|rumpun_ilmu|bidang_keilmuan|deskripsi_singkat|minat_cocok|mapel_relevan|hobi_relevan|gaya...,5,0
1,career,career_profile_text,nama_karier|bidang_karier|deskripsi_singkat|tugas_utama|skill_teknis|skill_non_teknis|tools_umum|prospek_industri|ke...,5,0
2,skill,skill_profile_text,nama_skill|kategori_skill|deskripsi_singkat|level_awal|prasyarat|sumber_belajar_awal|estimasi_waktu_belajar|keyword_...,7,0
3,roadmap,roadmap_profile_text,fase|durasi_rekomendasi|tujuan_fase|materi_pokok|aktivitas_praktik|output_portofolio|tools_umum|indikator_capaian,6,0


## 12. Export Dataset ke `data/processed` dan Report ke `reports/data_quality`

Cell ini menyimpan seluruh output Tahap 03.

In [16]:
# Export processed datasets
intent_out = DATA_PROCESSED_DIR / "intent_dataset_processed.csv"
program_out = DATA_PROCESSED_DIR / "program_studi_s1_processed.csv"
career_out = DATA_PROCESSED_DIR / "karier_processed.csv"
skill_out = DATA_PROCESSED_DIR / "skill_awal_processed.csv"
roadmap_out = DATA_PROCESSED_DIR / "roadmap_belajar_processed.csv"
normalization_out = DATA_PROCESSED_DIR / "normalisasi_bahasa_processed.csv"

intent_df.to_csv(intent_out, index=False, encoding="utf-8-sig")
program_df.to_csv(program_out, index=False, encoding="utf-8-sig")
career_df.to_csv(career_out, index=False, encoding="utf-8-sig")
skill_df.to_csv(skill_out, index=False, encoding="utf-8-sig")
roadmap_df.to_csv(roadmap_out, index=False, encoding="utf-8-sig")
norm_work.to_csv(normalization_out, index=False, encoding="utf-8-sig")

# Export reports
preprocessing_quality_out = REPORTS_DATA_QUALITY_DIR / "preprocessing_quality_summary.csv"
preprocessing_samples_out = REPORTS_DATA_QUALITY_DIR / "preprocessing_samples.csv"
feature_engineering_out = REPORTS_DATA_QUALITY_DIR / "feature_engineering_summary.csv"
preprocessing_validation_out = REPORTS_DATA_QUALITY_DIR / "preprocessing_validation_summary.csv"
text_audit_out = REPORTS_DATA_QUALITY_DIR / "text_audit_summary.csv"

preprocessing_quality_summary.to_csv(preprocessing_quality_out, index=False, encoding="utf-8-sig")
preprocessing_samples.to_csv(preprocessing_samples_out, index=False, encoding="utf-8-sig")
feature_engineering_summary.to_csv(feature_engineering_out, index=False, encoding="utf-8-sig")
preprocessing_validation_df.to_csv(preprocessing_validation_out, index=False, encoding="utf-8-sig")
text_audit_df.to_csv(text_audit_out, index=False, encoding="utf-8-sig")

saved_files = [
    intent_out,
    program_out,
    career_out,
    skill_out,
    roadmap_out,
    normalization_out,
    preprocessing_quality_out,
    preprocessing_samples_out,
    feature_engineering_out,
    preprocessing_validation_out,
    text_audit_out,
]

print("File berhasil disimpan:")
for path in saved_files:
    print("-", path.relative_to(PROJECT_ROOT))

File berhasil disimpan:
- data\processed\intent_dataset_processed.csv
- data\processed\program_studi_s1_processed.csv
- data\processed\karier_processed.csv
- data\processed\skill_awal_processed.csv
- data\processed\roadmap_belajar_processed.csv
- data\processed\normalisasi_bahasa_processed.csv
- reports\data_quality\preprocessing_quality_summary.csv
- reports\data_quality\preprocessing_samples.csv
- reports\data_quality\feature_engineering_summary.csv
- reports\data_quality\preprocessing_validation_summary.csv
- reports\data_quality\text_audit_summary.csv


## 13. Final Sanity Check

Cell ini memeriksa ulang apakah output utama benar-benar ada.

In [17]:
required_outputs = {
    "intent_processed": intent_out,
    "program_processed": program_out,
    "career_processed": career_out,
    "skill_processed": skill_out,
    "roadmap_processed": roadmap_out,
    "normalization_processed": normalization_out,
    "preprocessing_quality_summary": preprocessing_quality_out,
    "preprocessing_samples": preprocessing_samples_out,
    "feature_engineering_summary": feature_engineering_out,
}

sanity_check_df = pd.DataFrame([
    {
        "output_name": name,
        "relative_path": str(path.relative_to(PROJECT_ROOT)),
        "exists": path.exists(),
        "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else 0,
    }
    for name, path in required_outputs.items()
])

display(sanity_check_df)

if not sanity_check_df["exists"].all():
    missing = sanity_check_df.loc[~sanity_check_df["exists"], "relative_path"].tolist()
    raise FileNotFoundError(f"Masih ada output yang belum terbentuk: {missing}")

print("Sanity check selesai. Semua output utama Tahap 03 tersedia.")

,output_name,relative_path,exists,size_kb
0,intent_processed,data\processed\intent_dataset_processed.csv,True,5.74
1,program_processed,data\processed\program_studi_s1_processed.csv,True,6.20
2,career_processed,data\processed\karier_processed.csv,True,4.65
3,skill_processed,data\processed\skill_awal_processed.csv,True,5.09
4,roadmap_processed,data\processed\roadmap_belajar_processed.csv,True,4.74
5,normalization_processed,data\processed\normalisasi_bahasa_processed.csv,True,1.75
6,preprocessing_quality_summary,reports\data_quality\preprocessing_quality_summary.csv,True,0.67
7,preprocessing_samples,reports\data_quality\preprocessing_samples.csv,True,4.24
8,feature_engineering_summary,reports\data_quality\feature_engineering_summary.csv,True,0.78


Sanity check selesai. Semua output utama Tahap 03 tersedia.
